# 01 — Exploration des Données (EDA)
## Projet EDF — Prédiction de la Consommation Électrique Journalière

**Objectif :** Comprendre la structure et les patterns des données de consommation électrique française (source : RTE éco2mix) avant d'entraîner les modèles de prédiction.

**Dataset :** Consommation électrique journalière nationale 2019-2024 (~2200 observations)

**Plan du notebook :**
1. Chargement et génération des données
2. Statistiques descriptives
3. Analyse de la saisonnalité
4. Analyse des corrélations (température, type de jour)
5. Détection des outliers
6. Conclusions pour la modélisation

In [ ]:
# ── Imports ──
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

from data_pipeline.ingestion import generate_synthetic_data

# Style graphique
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

print('✅ Imports OK')

## 1. Chargement des données

In [ ]:
# Chargement des données (synthétiques si pas de fichier RTE disponible)
df = generate_synthetic_data(date_debut='2019-01-01', date_fin='2024-12-31')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

print(f'Shape du dataset : {df.shape}')
print(f'Période : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'\nColonnes : {list(df.columns)}')
df.head(10)

## 2. Statistiques descriptives

In [ ]:
stats_desc = df[['consommation_mw', 'temperature_moyenne',
                  'temperature_min', 'temperature_max']].describe().round(1)
print('=== Statistiques descriptives ===')
stats_desc

In [ ]:
# Distribution par type de jour
type_labels = {0: 'Ouvré', 1: 'Week-end', 2: 'Férié'}
df['type_label'] = df['type_jour'].map(type_labels)

print('Consommation moyenne par type de jour :')
print(df.groupby('type_label')['consommation_mw'].agg(['mean', 'std', 'count']).round(0))

## 3. Analyse de la saisonnalité

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Analyse de la Saisonnalité — Consommation Électrique EDF', fontsize=14, fontweight='bold')

# ── Panel 1 : Série temporelle complète
ax1 = axes[0, 0]
ax1.plot(df['date'], df['consommation_mw'] / 1000, alpha=0.6, linewidth=0.8, color='steelblue')
# Moyenne mobile 30 jours
ma30 = df['consommation_mw'].rolling(30, center=True).mean() / 1000
ax1.plot(df['date'], ma30, color='red', linewidth=2, label='MM 30j')
ax1.set_title('Série temporelle 2019-2024')
ax1.set_ylabel('Consommation (GW)')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax1.legend()

# ── Panel 2 : Profil mensuel moyen
ax2 = axes[0, 1]
mois_noms = ['Jan', 'Fév', 'Mar', 'Avr', 'Mai', 'Jun',
              'Jul', 'Aoû', 'Sep', 'Oct', 'Nov', 'Déc']
conso_mois = df.groupby('mois')['consommation_mw'].agg(['mean', 'std'])
ax2.bar(range(1, 13), conso_mois['mean'] / 1000, yerr=conso_mois['std'] / 1000,
        capsize=4, color='steelblue', alpha=0.8)
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(mois_noms, fontsize=9)
ax2.set_title('Profil mensuel moyen')
ax2.set_ylabel('Consommation (GW)')

# ── Panel 3 : Profil hebdomadaire
ax3 = axes[1, 0]
jours_noms = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']
conso_semaine = df.groupby('jour_semaine')['consommation_mw'].agg(['mean', 'std'])
colors = ['steelblue'] * 5 + ['coral'] * 2
ax3.bar(range(7), conso_semaine['mean'] / 1000, yerr=conso_semaine['std'] / 1000,
        capsize=4, color=colors, alpha=0.85)
ax3.set_xticks(range(7))
ax3.set_xticklabels(jours_noms)
ax3.set_title('Profil hebdomadaire moyen')
ax3.set_ylabel('Consommation (GW)')
ax3.axvspan(4.5, 6.5, alpha=0.1, color='red', label='Week-end')
ax3.legend()

# ── Panel 4 : Distribution de la consommation
ax4 = axes[1, 1]
ax4.hist(df['consommation_mw'] / 1000, bins=50, color='steelblue',
          alpha=0.8, edgecolor='white')
ax4.axvline(df['consommation_mw'].mean() / 1000, color='red',
             linestyle='--', label=f"Moy: {df['consommation_mw'].mean()/1000:.1f} GW")
ax4.set_title('Distribution de la consommation')
ax4.set_xlabel('Consommation (GW)')
ax4.set_ylabel('Fréquence')
ax4.legend()

plt.tight_layout()
plt.savefig('../data/processed/01_saisonnalite.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure sauvegardée.')

## 4. Analyse de la thermosensibilité

La consommation électrique française est fortement liée à la température en raison du **chauffage électrique** (~30% de la consommation nationale). On parle de **thermosensibilité** : chaque degré Celsius en dessous de 17°C entraîne une augmentation d'environ 2 400 MW de consommation supplémentaire.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Thermosensibilité — Relation Température / Consommation', fontsize=13, fontweight='bold')

# ── Scatter : Température vs Consommation
ax1 = axes[0]
scatter = ax1.scatter(df['temperature_moyenne'], df['consommation_mw'] / 1000,
                      c=df['mois'], cmap='coolwarm', alpha=0.5, s=8)
plt.colorbar(scatter, ax=ax1, label='Mois')

# Régression linéaire
mask_froid = df['temperature_moyenne'] < 17
slope, intercept, r, p, se = stats.linregress(
    df.loc[mask_froid, 'temperature_moyenne'],
    df.loc[mask_froid, 'consommation_mw'] / 1000
)
x_line = np.linspace(df['temperature_moyenne'].min(), 17, 50)
ax1.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2,
          label=f'Régression (T<17°C)\nPente: {slope:.1f} GW/°C, R²={r**2:.3f}')
ax1.axvline(17, color='green', linestyle='--', alpha=0.7, label='Seuil thermosensibilité (17°C)')
ax1.set_xlabel('Température moyenne (°C)')
ax1.set_ylabel('Consommation (GW)')
ax1.set_title('Scatter : Température vs Consommation')
ax1.legend(fontsize=9)

# ── Corrélation par mois
ax2 = axes[1]
corr_mois = df.groupby('mois').apply(
    lambda g: g['temperature_moyenne'].corr(g['consommation_mw'])
).reset_index()
corr_mois.columns = ['mois', 'correlation']
colors_corr = ['red' if c < 0 else 'steelblue' for c in corr_mois['correlation']]
ax2.bar(corr_mois['mois'], corr_mois['correlation'], color=colors_corr, alpha=0.85)
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(mois_noms, fontsize=9)
ax2.set_title('Corrélation Température-Consommation par mois')
ax2.set_ylabel('Coefficient de corrélation')
ax2.axhline(0, color='black', linewidth=0.8)

print(f'Thermosensibilité mesurée (T < 17°C) : {slope*1000:.0f} MW/°C')
print(f'(Valeur réelle EDF : ~2 400 MW/°C)')

plt.tight_layout()
plt.savefig('../data/processed/02_thermosensibilite.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Matrice de corrélation

In [ ]:
# Calcul des lags pour la matrice de corrélation
df_corr = df.copy()
df_corr['conso_j1'] = df_corr['consommation_mw'].shift(1)
df_corr['conso_j7'] = df_corr['consommation_mw'].shift(7)
df_corr = df_corr.dropna()

cols_corr = ['consommation_mw', 'conso_j1', 'conso_j7',
              'temperature_moyenne', 'type_jour', 'mois', 'jour_semaine']

corr_matrix = df_corr[cols_corr].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, mask=mask, ax=ax,
            xticklabels=['Conso J', 'Conso J-1', 'Conso J-7', 'Temp.', 'Type jour', 'Mois', 'Jour sem.'],
            yticklabels=['Conso J', 'Conso J-1', 'Conso J-7', 'Temp.', 'Type jour', 'Mois', 'Jour sem.'])
ax.set_title('Matrice de corrélation des variables', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.savefig('../data/processed/03_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

# Corrélations avec la cible
print('\nCorrélations avec la consommation du jour :')
print(corr_matrix['consommation_mw'].sort_values(ascending=False).drop('consommation_mw'))

## 6. Détection des outliers

In [ ]:
# Méthode IQR
Q1 = df['consommation_mw'].quantile(0.25)
Q3 = df['consommation_mw'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 3 * IQR
upper_bound = Q3 + 3 * IQR

outliers = df[(df['consommation_mw'] < lower_bound) | (df['consommation_mw'] > upper_bound)]

print(f'Seuil bas  : {lower_bound:,.0f} MW')
print(f'Seuil haut : {upper_bound:,.0f} MW')
print(f'Outliers détectés : {len(outliers)} ({len(outliers)/len(df)*100:.2f}% du dataset)')

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df['date'], df['consommation_mw'] / 1000, linewidth=0.6, color='steelblue', alpha=0.7)
if len(outliers) > 0:
    ax.scatter(outliers['date'], outliers['consommation_mw'] / 1000,
               color='red', s=30, zorder=5, label=f'Outliers ({len(outliers)})')
ax.axhline(lower_bound / 1000, color='orange', linestyle='--', alpha=0.7, label='Seuil IQR bas')
ax.axhline(upper_bound / 1000, color='orange', linestyle='--', alpha=0.7, label='Seuil IQR haut')
ax.set_ylabel('Consommation (GW)')
ax.set_title('Détection des outliers (méthode 3×IQR)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('../data/processed/04_outliers.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Conclusions pour la modélisation

L'analyse exploratoire révèle les points clés suivants :

| Observation | Impact sur la modélisation |
|---|---|
| **Forte autocorrélation J-1 et J-7** | Inclure `consommation_j1` et `consommation_j7` comme features principales |
| **Thermosensibilité forte (T < 17°C)** | Inclure `temperature_moyenne`, `temperature_min`, `temperature_max` |
| **Creux week-end (-8%) et férié (-12%)** | Encoder `type_jour` avec One-Hot Encoding |
| **Saisonnalité annuelle marquée** | Utiliser `mois_sin` / `mois_cos` (encodage cyclique) |
| **Pas d'outliers extrêmes** | Nettoyage minimal nécessaire |
| **Distribution légèrement bimodale** | Les modèles non-paramétriques (RF, KNN) adaptés |

**Features retenues pour l'entraînement :** 14 variables (cf. `preprocessing.py`)

In [ ]:
print('=== RÉSUMÉ EDA ===')
print(f'Période analysée  : {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Observations      : {len(df)}')
print(f'Conso. moyenne    : {df["consommation_mw"].mean():,.0f} MW ({df["consommation_mw"].mean()/1000:.1f} GW)')
print(f'Conso. min        : {df["consommation_mw"].min():,.0f} MW')
print(f'Conso. max        : {df["consommation_mw"].max():,.0f} MW')
print(f'Écart-type        : {df["consommation_mw"].std():,.0f} MW')
print(f'Corrélation J-1   : {df["consommation_mw"].corr(df["consommation_mw"].shift(1)):.3f}')
print(f'Corrélation Temp. : {df["consommation_mw"].corr(df["temperature_moyenne"]):.3f}')
print('\n✅ EDA terminé — Prêt pour le preprocessing (notebook 02)')